In [1]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
token = user_secrets.get_secret("hugging-face")

login(token)

# Remova qualquer versão antiga
!pip uninstall -y diffusers

# Instale direto do GitHub (source install)
!pip install git+https://github.com/huggingface/diffusers


!pip install git+https://github.com/huggingface/accelerate
!pip install git+https://github.com/huggingface/transformers
!pip install peft datasets torchao -U


!git clone https://github.com/huggingface/diffusers
%cd diffusers/examples/text_to_image

Found existing installation: diffusers 0.37.1
Uninstalling diffusers-0.37.1:
  Successfully uninstalled diffusers-0.37.1
  Cloning https://github.com/huggingface/diffusers to /tmp/pip-req-build-mgcp0g5n
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/diffusers /tmp/pip-req-build-mgcp0g5n
  Resolved https://github.com/huggingface/diffusers to commit 284419b35267dea7d4435fd0ef1eaa132bf73efe
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 26.7 MB/s eta 0:00:00
  Created wheel for diffusers: filename=diffusers-0.40.0.dev0-py3-none-any.whl size=5648379 sha256=7b4f0c42e805583c41d82d6a6f11e93d9e57d0467315ff7cc22c9e9dab30e463
  Stored in directory: /tmp/pip-ephem-wheel-cache-wi81d_vp/wheels/90/d4/44/a58bc00fb405fefb633b0d9d2307f6e3aec6cc1775d82555d3
Successfully built diffusers
  Attempting uninstall: safetensors

In [ ]:
# # 1. Certifique-se de estar na pasta raiz do trabalho
# %cd /kaggle/working/

# # 2. Se a pasta diffusers já existir de uma tentativa anterior, remova para evitar conflitos
# !rm -rf diffusers

# # 3. Clone novamente
# !git clone https://github.com/huggingface/diffusers

# # 4. Entre na pasta raiz do projeto (onde fica o pyproject.toml)
# %cd diffusers

# # 5. Instale o pacote a partir desta pasta
# # !pip install -q diffusers transformers accelerate peft datasets torchao -U

# # 6. Agora sim, entre na pasta dos exemplos
# %cd examples/text_to_image

In [2]:
import pandas as pd
import os

# 1. Clone o repositório (direcionando para o ambiente Kaggle)
!git clone https://github.com/hpsj2712/atelie-generativo-individual-homerio /kaggle/working/temp_repo

# 2. Prepare o diretório do dataset
!mkdir -p /kaggle/working/dataset

# 3. Mova as imagens para a pasta final
!mv /kaggle/working/temp_repo/dados/*.jpg /kaggle/working/dataset/

# 4. Processar as legendas (legendas.txt)
legendas_dict = {}
# Ajustado para ler do caminho correto no Kaggle
with open('/kaggle/working/temp_repo/dados/legendas.txt', 'r', encoding='utf-8') as f:
    for line in f:
        if ":" in line:
            parts = line.split(":", 1)
            file_name = parts[0].strip()
            caption = parts[1].strip()
            legendas_dict[file_name] = caption

# 5. Criar o metadata.csv no formato exigido pelo Diffusers
df = pd.DataFrame(list(legendas_dict.items()), columns=['file_name', 'text'])
df.to_csv('/kaggle/working/dataset/metadata.csv', index=False)

print("Dataset e metadata.csv criados com sucesso em /kaggle/working/dataset/")
!rm -rf /kaggle/working/temp_repo

Cloning into '/kaggle/working/temp_repo'...
remote: Enumerating objects: 192, done.
remote: Counting objects: 100% (192/192), done.
remote: Compressing objects: 100% (171/171), done.
remote: Total 192 (delta 61), reused 40 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (192/192), 138.39 MiB | 28.08 MiB/s, done.
Resolving deltas: 100% (61/61), done.
Dataset e metadata.csv criados com sucesso em /kaggle/working/dataset/


configurar acelerate

Primeiro treino: 
Via Kagge sem acelerate
Gerou imagens condizentes com o treino.

In [ ]:
# !accelerate launch train_text_to_image_lora.py \
!python train_text_to_image_lora.py \
  --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
  --train_data_dir="/kaggle/working/dataset" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --max_train_steps=1500 \
  --learning_rate=1e-4 \
  --lr_scheduler="cosine" \
  --rank=8 \
  --mixed_precision="fp16" \
  --checkpointing_steps=500 \
  --validation_prompt="estilo_arquitetura_modernista_brasilia, um predio oficial no inverno" \
  --output_dir="/kaggle/working/lora_saida" \
  --push_to_hub \
  --hub_model_id="homerio/estilo_arquitetura_modernista_brasilia"

In [ ]:
!accelerate launch train_text_to_image_lora.py \
  --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
  --train_data_dir="/content/dataset" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --max_train_steps=1000 \
  --learning_rate=3e-4 \
  --lr_scheduler="cosine" \
  --rank=5 \
  --mixed_precision="fp16" \
  --checkpointing_steps=200 \
  --validation_prompt="estilo_arquitetura_modernista_brasilia, a concret square with a museum" \
  --output_dir="/content/lora_saida" \
  --push_to_hub \
  --hub_model_id="homerio/estilo_arquitetura_modernista_brasilia_v1"

In [ ]:
# !accelerate launch --num_processes=1 train_text_to_image_lora.py \
!python train_text_to_image_lora.py \
  --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
  --train_data_dir="/kaggle/working/dataset" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --max_train_steps=1000 \
  --learning_rate=5e-5 \
  --lr_scheduler="cosine" \
  --rank=8 \
  --mixed_precision="fp16" \
  --checkpointing_steps=200 \
  --validation_prompt="estilo_arquitetura_modernista_brasilia, a concret square with a museum" \
  --output_dir="/kaggle/working/lora_saida" \
  --push_to_hub \
  --hub_model_id="homerio/estilo_arquitetura_modernista_brasilia_v2"

V3:

In [ ]:
# !accelerate launch --num_processes=1 train_text_to_image_lora.py \
!python train_text_to_image_lora.py \
  --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
  --train_data_dir="/kaggle/working/dataset" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --max_train_steps=1000 \
  --learning_rate=1e-5 \
  --lr_scheduler="cosine" \
  --rank=7 \
  --mixed_precision="fp16" \
  --checkpointing_steps=200 \
  --validation_prompt="estilo_arquitetura_modernista_brasilia, a concret square with a museum" \
  --output_dir="/kaggle/working/lora_saida" \
  --push_to_hub \
  --hub_model_id="homerio/estilo_arquitetura_modernista_brasilia_v3"

V4:

In [17]:
# !accelerate launch train_text_to_image_lora.py \
!python train_text_to_image_lora.py \
  --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
  --train_data_dir="/kaggle/working/dataset" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --max_train_steps=1500 \
  --learning_rate=2e-4 \
  --lr_scheduler="cosine" \
  --rank=16 \
  --mixed_precision="fp16" \
  --checkpointing_steps=500 \
  --validation_prompt="estilo_arquitetura_modernista_brasilia, um museu numa praça de concrento num dia ensolarado" \
  --output_dir="/kaggle/working/lora_saida" \
  --push_to_hub \
  --hub_model_id="homerio/estilo_arquitetura_modernista_brasilia_v4"

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
07/10/2026 02:56:03 - INFO - __main__ - [RANK 0] Distributed environment: DistributedType.NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: fp16

07/10/2026 02:56:03 - INFO - httpx - HTTP Request: POST https://huggingface.co/api/repos/create "HTTP/1.1 409 Conflict"
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored